In [1]:
import os
import math
import json
import random
from glob import glob

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

In [ ]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", DEVICE)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# -------- paths --------
CLEAN_DIR = r"D:\Masters\Speech\Embeddings\clean"   
PAIR_DIR  = r"D:\Masters\Speech\articulation_pairs_pronounced"  
MODEL_PATH = r"D:\Masters\Speech\reverse_articulation_pronounced_best.pt"

# -------- forward diffusion settings --------
T_STEPS = 30
SCHEDULE = "cosine"

FORWARD_CFG = {
    "total_scale": 1.30,          
    "noise_smooth_kernel": 9,
    "embed_smooth_kernel": 7,
    "max_blur_alpha": 0.85,
    "max_mix_prob": 0.65,
    "max_mix_lam": 0.70,
    "span_density_base": 0.04,
    "span_density_scale": 0.16,
    "span_len_min": 3,
    "span_len_max": 9,
    "hard_replace_prob": 0.22,    
    "transition_power": 1.75,
    "weakening_strength": 0.18,   
}


BATCH_SIZE = 4
EPOCHS = 20
LR = 1e-4
NUM_WORKERS = 0
GRAD_CLIP = 1.0

# -------- stats --------
# Replace with your measured real stats if you have them
ARTIC_STATS = {
    "drift_l2_mean": 1.2,
    "drift_l2_std": 0.45,
    "drift_l2_p95": 2.2,
}

Using device: cpu


In [ ]:
def moving_average_1d(x: np.ndarray, kernel_size: int = 5) -> np.ndarray:
    """
    Moving average along time axis for shape (T, D).
    """
    if kernel_size <= 1:
        return x.astype(np.float32)

    x = x.astype(np.float32)
    T, D = x.shape
    pad = kernel_size // 2
    x_pad = np.pad(x, ((pad, pad), (0, 0)), mode="edge")

    out = np.zeros_like(x, dtype=np.float32)
    for t in range(T):
        out[t] = np.mean(x_pad[t:t + kernel_size], axis=0)
    return out.astype(np.float32)


def get_step_weights(T: int, schedule: str = "cosine") -> np.ndarray:
    """
    Returns normalized weights summing to 1.
    Later steps stronger for linear/cosine.
    """
    if schedule == "uniform":
        w = np.ones(T, dtype=np.float32)
    elif schedule == "linear":
        w = np.arange(1, T + 1, dtype=np.float32)
    elif schedule == "cosine":
        t = np.linspace(0, 1, T, dtype=np.float32)
        w = 1.0 - np.cos((t + 1e-8) * np.pi / 2.0)
    else:
        raise ValueError(f"Unknown schedule: {schedule}")

    w = np.maximum(w, 1e-8)
    w = w / np.sum(w)
    return w.astype(np.float32)


def cosine_similarity_mean(a: np.ndarray, b: np.ndarray, eps: float = 1e-8) -> float:
    a = a.astype(np.float32)
    b = b.astype(np.float32)
    num = np.sum(a * b, axis=1)
    den = np.linalg.norm(a, axis=1) * np.linalg.norm(b, axis=1) + eps
    return float(np.mean(num / den))


def evaluate_pair(clean: np.ndarray, corrupted: np.ndarray) -> dict:
    diff = corrupted - clean
    return {
        "mean_abs_error": float(np.mean(np.abs(diff))),
        "max_abs_error": float(np.max(np.abs(diff))),
        "mean_l2_per_frame": float(np.mean(np.linalg.norm(diff, axis=1))),
        "cosine": cosine_similarity_mean(clean, corrupted),
    }

In [ ]:
def articulation_forward_step_pronounced(
    x_prev: np.ndarray,
    stats: dict,
    step_weight: float,
    total_scale: float = 1.30,
    noise_smooth_kernel: int = 9,
    embed_smooth_kernel: int = 7,
    max_blur_alpha: float = 0.85,
    max_mix_prob: float = 0.65,
    max_mix_lam: float = 0.70,
    span_density_base: float = 0.04,
    span_density_scale: float = 0.16,
    span_len_min: int = 3,
    span_len_max: int = 9,
    hard_replace_prob: float = 0.22,
    hard_replace_lam: float = 0.85,
    transition_power: float = 1.75,
    weakening_strength: float = 0.18,
    seed: int = 42
):
    """
    Pronounced articulatory corruption in embedding space.

    Key idea:
      - target transition-heavy regions
      - corrupt short spans
      - combine drift + blur + frame confusion + partial replacement
      - selectively weaken high-change regions
    """
    rng = np.random.default_rng(seed)

    x_prev = x_prev.astype(np.float32)
    Tseq, D = x_prev.shape

    drift_mean = float(stats.get("drift_l2_mean", 1.0))
    drift_std  = float(stats.get("drift_l2_std", 0.5))
    drift_p95  = float(stats.get("drift_l2_p95", drift_mean + 2.0 * drift_std))


    frame_diff = np.linalg.norm(x_prev[1:] - x_prev[:-1], axis=1)
    frame_diff = np.pad(frame_diff, (1, 0), mode="edge").astype(np.float32)

    transition_score = np.power(frame_diff + 1e-8, transition_power)
    transition_prob = transition_score / np.sum(transition_score)

    num_spans = max(2, int((span_density_base + span_density_scale * step_weight) * Tseq))
    span_mask = np.zeros(Tseq, dtype=np.float32)

    for _ in range(num_spans):
        center = int(rng.choice(Tseq, p=transition_prob))
        span_len = int(rng.integers(span_len_min, span_len_max + 1))
        start = max(0, center - span_len // 2)
        end = min(Tseq, start + span_len)
        span_mask[start:end] = 1.0

    span_mask_2d = span_mask[:, None]


    raw_noise = rng.normal(0.0, 1.0, size=(Tseq, D)).astype(np.float32)
    smooth_noise = moving_average_1d(raw_noise, kernel_size=noise_smooth_kernel)

    norms = np.linalg.norm(smooth_noise, axis=1, keepdims=True) + 1e-8
    direction = smooth_noise / norms

    mag = rng.normal(
        loc=drift_mean,
        scale=max(drift_std, 1e-6),
        size=(Tseq, 1)
    ).astype(np.float32)
    mag = np.clip(mag, 0.0, drift_p95)

    # more aggressive later in diffusion
    mag *= total_scale * (0.45 + 2.0 * step_weight)

    delta_t = span_mask_2d * (mag * direction)
    x_t = x_prev + delta_t

  
    blur_alpha = max_blur_alpha * (0.35 + 0.65 * step_weight)
    x_smooth = moving_average_1d(x_t, kernel_size=embed_smooth_kernel)
    x_t = x_t + span_mask_2d * (blur_alpha * (x_smooth - x_t))


    mix_prob = max_mix_prob * (0.35 + 0.65 * step_weight)
    mix_lam  = max_mix_lam  * (0.35 + 0.65 * step_weight)

    shift = int(rng.choice([-3, -2, -1, 1, 2, 3]))
    x_shift = np.roll(x_prev, shift=shift, axis=0)

    mix_gate = (rng.random((Tseq, 1)) < mix_prob).astype(np.float32) * span_mask_2d
    x_t = x_t + mix_gate * (mix_lam * (x_shift - x_t))

    hr_prob = hard_replace_prob * (0.35 + 0.65 * step_weight)
    hr_lam  = hard_replace_lam  * (0.35 + 0.65 * step_weight)

    shift2 = int(rng.choice([-2, -1, 1, 2]))
    x_shift2 = np.roll(x_prev, shift=shift2, axis=0)
    hard_gate = (rng.random((Tseq, 1)) < hr_prob).astype(np.float32) * span_mask_2d
    x_t = x_t + hard_gate * (hr_lam * (x_shift2 - x_t))

    
    norm_diff = frame_diff / (np.max(frame_diff) + 1e-8)
    weaken = 1.0 - weakening_strength * norm_diff[:, None] * span_mask_2d
    x_t = x_t * weaken + x_prev * (1.0 - weaken) * 0.15

    delta_t = x_t - x_prev

    meta = {
        "step_weight": float(step_weight),
        "num_spans": int(num_spans),
        "corrupted_fraction": float(np.mean(span_mask)),
        "delta_mean_l2": float(np.mean(np.linalg.norm(delta_t, axis=1))),
        "delta_max_l2": float(np.max(np.linalg.norm(delta_t, axis=1))),
        "blur_alpha": float(blur_alpha),
        "mix_prob": float(mix_prob),
        "mix_lam": float(mix_lam),
        "hr_prob": float(hr_prob),
        "hr_lam": float(hr_lam),
    }

    return x_t.astype(np.float32), delta_t.astype(np.float32), meta

In [ ]:
def articulation_forward_diffusion_chain_pronounced(
    x_clean: np.ndarray,
    stats: dict,
    T: int = 30,
    schedule: str = "cosine",
    seed: int = 42,
    **forward_cfg
):
    """
    Builds x_0 -> x_1 -> ... -> x_T
    """
    x_clean = x_clean.astype(np.float32)
    x_states = [x_clean.copy()]
    deltas = [None]
    metas = [None]

    x = x_clean.copy()
    step_weights = get_step_weights(T, schedule=schedule)

    for t in range(1, T + 1):
        x, delta_t, meta_t = articulation_forward_step_pronounced(
            x_prev=x,
            stats=stats,
            step_weight=float(step_weights[t - 1]),
            seed=seed + t,
            **forward_cfg
        )
        meta_t["t"] = t
        x_states.append(x)
        deltas.append(delta_t)
        metas.append(meta_t)

    return x_states, deltas, metas

In [ ]:
sample_files = sorted(glob(os.path.join(CLEAN_DIR, "*.npy")))
print("Found:", len(sample_files))

x_clean = np.load(sample_files[0]).astype(np.float32)

x_states, deltas, metas = articulation_forward_diffusion_chain_pronounced(
    x_clean=x_clean,
    stats=ARTIC_STATS,
    T=T_STEPS,
    schedule=SCHEDULE,
    seed=SEED,
    **FORWARD_CFG
)

x_T = x_states[-1]

print("clean -> x_T:", evaluate_pair(x_clean, x_T))
print("last meta:", metas[-1])

In [ ]:
def make_reverse_pairs_from_clean_folder(
    clean_dir: str,
    pair_dir: str,
    stats: dict,
    T: int = 30,
    schedule: str = "cosine",
    seed: int = 42,
    **forward_cfg
):
    """
    For each clean file:
      generate x_0..x_T
      save training pairs:
        input  = x_t
        target = x_{t-1}
    """
    os.makedirs(pair_dir, exist_ok=True)

    files = sorted(glob(os.path.join(clean_dir, "*.npy")))
    print("Number of clean files:", len(files))

    manifest = []

    for i, fpath in enumerate(files, 1):
        x_clean = np.load(fpath).astype(np.float32)

        x_states, deltas, metas = articulation_forward_diffusion_chain_pronounced(
            x_clean=x_clean,
            stats=stats,
            T=T,
            schedule=schedule,
            seed=seed + i,
            **forward_cfg
        )

        base = os.path.splitext(os.path.basename(fpath))[0]

        for t in range(1, T + 1):
            save_path = os.path.join(pair_dir, f"{base}_t{t:02d}.npz")
            np.savez_compressed(
                save_path,
                x_t=x_states[t].astype(np.float32),
                x_prev=x_states[t - 1].astype(np.float32),
                t=np.array([t], dtype=np.int64),
                T=np.array([T], dtype=np.int64),
            )
            manifest.append(save_path)

        summary = evaluate_pair(x_clean, x_states[-1])
        print(
            f"[{i:04d}/{len(files):04d}] {base} | "
            f"clean->x_T cosine={summary['cosine']:.6f} | "
            f"mae={summary['mean_abs_error']:.6f}"
        )

    manifest_path = os.path.join(pair_dir, "manifest.json")
    with open(manifest_path, "w") as f:
        json.dump(manifest, f, indent=2)

    print("Saved manifest to:", manifest_path)
    print("Total training pairs:", len(manifest))

In [ ]:
make_reverse_pairs_from_clean_folder(
    clean_dir=CLEAN_DIR,
    pair_dir=PAIR_DIR,
    stats=ARTIC_STATS,
    T=T_STEPS,
    schedule=SCHEDULE,
    seed=SEED,
    **FORWARD_CFG
)

In [ ]:
class ReverseArticulationDataset(Dataset):
    def __init__(self, pair_dir: str):
        manifest_path = os.path.join(pair_dir, "manifest.json")
        with open(manifest_path, "r") as f:
            self.files = json.load(f)

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        item = np.load(self.files[idx])
        x_t = item["x_t"].astype(np.float32)       # (Tseq, D)
        x_prev = item["x_prev"].astype(np.float32) # (Tseq, D)
        t = int(item["t"][0])
        T = int(item["T"][0])

        return {
            "x_t": x_t,
            "x_prev": x_prev,
            "t": t,
            "T": T,
            "length": x_t.shape[0]
        }


def collate_reverse(batch):
    max_len = max(x["length"] for x in batch)
    D = batch[0]["x_t"].shape[1]

    x_t_batch = np.zeros((len(batch), max_len, D), dtype=np.float32)
    x_prev_batch = np.zeros((len(batch), max_len, D), dtype=np.float32)
    mask_batch = np.zeros((len(batch), max_len), dtype=np.float32)
    t_batch = np.zeros((len(batch),), dtype=np.int64)
    T_batch = np.zeros((len(batch),), dtype=np.int64)

    for i, item in enumerate(batch):
        L = item["length"]
        x_t_batch[i, :L] = item["x_t"]
        x_prev_batch[i, :L] = item["x_prev"]
        mask_batch[i, :L] = 1.0
        t_batch[i] = item["t"]
        T_batch[i] = item["T"]

    return {
        "x_t": torch.from_numpy(x_t_batch),         # (B, L, D)
        "x_prev": torch.from_numpy(x_prev_batch),   # (B, L, D)
        "mask": torch.from_numpy(mask_batch),       # (B, L)
        "t": torch.from_numpy(t_batch),             # (B,)
        "T": torch.from_numpy(T_batch),             # (B,)
    }

In [ ]:
class SinusoidalTimeEmbedding(nn.Module):
    def __init__(self, dim: int):
        super().__init__()
        self.dim = dim

    def forward(self, t_norm: torch.Tensor):
        """
        t_norm: (B,) in [0,1]
        returns: (B, dim)
        """
        device = t_norm.device
        half = self.dim // 2
        freqs = torch.exp(
            -math.log(10000) * torch.arange(0, half, device=device).float() / max(half - 1, 1)
        )
        args = t_norm[:, None] * freqs[None, :]
        emb = torch.cat([torch.sin(args), torch.cos(args)], dim=-1)
        if self.dim % 2 == 1:
            emb = F.pad(emb, (0, 1))
        return emb


class ResidualBlock1D(nn.Module):
    def __init__(self, channels: int, kernel_size: int = 5, dilation: int = 1):
        super().__init__()
        pad = (kernel_size - 1) // 2 * dilation
        self.conv1 = nn.Conv1d(channels, channels, kernel_size, padding=pad, dilation=dilation)
        self.conv2 = nn.Conv1d(channels, channels, kernel_size, padding=pad, dilation=dilation)
        self.norm1 = nn.GroupNorm(8, channels)
        self.norm2 = nn.GroupNorm(8, channels)

    def forward(self, x):
        res = x
        x = self.conv1(x)
        x = self.norm1(x)
        x = F.gelu(x)
        x = self.conv2(x)
        x = self.norm2(x)
        x = F.gelu(x)
        return x + res


class ReverseArticulationUNetLite(nn.Module):
    def __init__(self, dim_in=1024, hidden=384, time_dim=128, n_blocks=6):
        super().__init__()

        self.time_embed = SinusoidalTimeEmbedding(time_dim)
        self.time_mlp = nn.Sequential(
            nn.Linear(time_dim, hidden),
            nn.GELU(),
            nn.Linear(hidden, hidden)
        )

        self.in_proj = nn.Conv1d(dim_in, hidden, kernel_size=1)

        self.blocks = nn.ModuleList([
            ResidualBlock1D(hidden, kernel_size=5, dilation=1),
            ResidualBlock1D(hidden, kernel_size=5, dilation=2),
            ResidualBlock1D(hidden, kernel_size=5, dilation=4),
            ResidualBlock1D(hidden, kernel_size=5, dilation=1),
            ResidualBlock1D(hidden, kernel_size=5, dilation=2),
            ResidualBlock1D(hidden, kernel_size=5, dilation=4),
        ])

        self.out_proj = nn.Conv1d(hidden, dim_in, kernel_size=1)

    def forward(self, x_t, t, T):
        """
        x_t: (B, L, D)
        t:   (B,)
        T:   (B,)
        returns x_prev_hat: (B, L, D)
        """
        B, L, D = x_t.shape

        t_norm = t.float() / T.float().clamp(min=1.0)
        t_emb = self.time_embed(t_norm)          # (B, time_dim)
        t_emb = self.time_mlp(t_emb)             # (B, hidden)

        x = x_t.transpose(1, 2)                  # (B, D, L)
        x = self.in_proj(x)                      # (B, hidden, L)

        # add time conditioning
        x = x + t_emb[:, :, None]

        for blk in self.blocks:
            x = blk(x)

        delta_hat = self.out_proj(x).transpose(1, 2)  # (B, L, D)

        # residual prediction: predict correction from x_t to x_{t-1}
        x_prev_hat = x_t + delta_hat
        return x_prev_hat

In [ ]:
def masked_l1_l2_loss(pred, target, mask, l1_weight=1.0, l2_weight=1.0):
    """
    pred,target: (B,L,D)
    mask: (B,L)
    """
    mask = mask.unsqueeze(-1)  # (B,L,1)

    l1 = torch.abs(pred - target) * mask
    l2 = (pred - target) ** 2 * mask

    denom = mask.sum().clamp(min=1.0) * pred.shape[-1]
    l1 = l1.sum() / denom
    l2 = l2.sum() / denom

    return l1_weight * l1 + l2_weight * l2, l1.detach(), l2.detach()


dataset = ReverseArticulationDataset(PAIR_DIR)
n_total = len(dataset)
n_train = int(0.9 * n_total)
n_val = n_total - n_train

train_set, val_set = torch.utils.data.random_split(
    dataset,
    [n_train, n_val],
    generator=torch.Generator().manual_seed(SEED)
)

train_loader = DataLoader(
    train_set,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    collate_fn=collate_reverse
)

val_loader = DataLoader(
    val_set,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    collate_fn=collate_reverse
)

print("Train pairs:", len(train_set))
print("Val pairs:", len(val_set))

In [ ]:
model = ReverseArticulationUNetLite(dim_in=1024, hidden=384, time_dim=128).to(DEVICE)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)

best_val = float("inf")

for epoch in range(1, EPOCHS + 1):
    # ---------------- train ----------------
    model.train()
    train_loss_sum = 0.0
    train_l1_sum = 0.0
    train_l2_sum = 0.0
    n_train_batches = 0

    for batch in train_loader:
        x_t = batch["x_t"].to(DEVICE)
        x_prev = batch["x_prev"].to(DEVICE)
        mask = batch["mask"].to(DEVICE)
        t = batch["t"].to(DEVICE)
        T = batch["T"].to(DEVICE)

        optimizer.zero_grad()

        pred = model(x_t, t, T)
        loss, l1_part, l2_part = masked_l1_l2_loss(pred, x_prev, mask, l1_weight=1.0, l2_weight=1.0)

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
        optimizer.step()

        train_loss_sum += loss.item()
        train_l1_sum += l1_part.item()
        train_l2_sum += l2_part.item()
        n_train_batches += 1

    train_loss = train_loss_sum / max(n_train_batches, 1)
    train_l1 = train_l1_sum / max(n_train_batches, 1)
    train_l2 = train_l2_sum / max(n_train_batches, 1)

    # ---------------- val ----------------
    model.eval()
    val_loss_sum = 0.0
    val_l1_sum = 0.0
    val_l2_sum = 0.0
    n_val_batches = 0

    with torch.no_grad():
        for batch in val_loader:
            x_t = batch["x_t"].to(DEVICE)
            x_prev = batch["x_prev"].to(DEVICE)
            mask = batch["mask"].to(DEVICE)
            t = batch["t"].to(DEVICE)
            T = batch["T"].to(DEVICE)

            pred = model(x_t, t, T)
            loss, l1_part, l2_part = masked_l1_l2_loss(pred, x_prev, mask, l1_weight=1.0, l2_weight=1.0)

            val_loss_sum += loss.item()
            val_l1_sum += l1_part.item()
            val_l2_sum += l2_part.item()
            n_val_batches += 1

    val_loss = val_loss_sum / max(n_val_batches, 1)
    val_l1 = val_l1_sum / max(n_val_batches, 1)
    val_l2 = val_l2_sum / max(n_val_batches, 1)

    print(
        f"Epoch {epoch:02d} | "
        f"train={train_loss:.6f} | train_l1={train_l1:.6f} | train_l2={train_l2:.6f} | "
        f"val={val_loss:.6f} | val_l1={val_l1:.6f} | val_l2={val_l2:.6f}"
    )

    if val_loss < best_val:
        best_val = val_loss
        torch.save(model.state_dict(), MODEL_PATH)
        print("  Saved best model to", MODEL_PATH)

In [ ]:
best_model = ReverseArticulationUNetLite(dim_in=1024, hidden=384, time_dim=128).to(DEVICE)
best_model.load_state_dict(torch.load(MODEL_PATH, map_location=DEVICE))
best_model.eval()

print("Loaded:", MODEL_PATH)

In [ ]:
@torch.no_grad()
def reverse_articulation_diffusion(
    model: nn.Module,
    x_T: np.ndarray,
    T_steps: int,
    device: str = "cpu"
):
    """
    Iteratively denoise:
      x_T -> x_{T-1} -> ... -> x_0_hat
    """
    x = torch.from_numpy(x_T.astype(np.float32)).unsqueeze(0).to(device)  # (1,L,D)

    for t_cur in range(T_steps, 0, -1):
        t_tensor = torch.tensor([t_cur], dtype=torch.long, device=device)
        T_tensor = torch.tensor([T_steps], dtype=torch.long, device=device)

        x = model(x, t_tensor, T_tensor)

    return x.squeeze(0).cpu().numpy().astype(np.float32)

In [ ]:
sample_files = sorted(glob(os.path.join(CLEAN_DIR, "*.npy")))
x_clean = np.load(sample_files[0]).astype(np.float32)

# forward corruption
x_states, deltas, metas = articulation_forward_diffusion_chain_pronounced(
    x_clean=x_clean,
    stats=ARTIC_STATS,
    T=T_STEPS,
    schedule=SCHEDULE,
    seed=SEED,
    **FORWARD_CFG
)

x_T = x_states[-1]

# reverse recovery
x_recovered = reverse_articulation_diffusion(
    model=best_model,
    x_T=x_T,
    T_steps=T_STEPS,
    device=DEVICE
)

print("clean -> x_T cosine:", cosine_similarity_mean(x_clean, x_T))
print("clean -> recovered cosine:", cosine_similarity_mean(x_clean, x_recovered))
print("recovered mean abs error:", float(np.mean(np.abs(x_clean - x_recovered))))
print("recovered max abs error:", float(np.max(np.abs(x_clean - x_recovered))))

In [ ]:
identity_mae = float(np.mean(np.abs(x_clean - x_T)))
model_mae = float(np.mean(np.abs(x_clean - x_recovered)))

identity_cos = cosine_similarity_mean(x_clean, x_T)
model_cos = cosine_similarity_mean(x_clean, x_recovered)

print("Identity baseline MAE:", identity_mae)
print("Model recovery MAE:", model_mae)
print("Identity baseline cosine:", identity_cos)
print("Model recovery cosine:", model_cos)

In [ ]:
test_files = sorted(glob(os.path.join(CLEAN_DIR, "*.npy")))[:20]

all_results = []

for i, fpath in enumerate(test_files, 1):
    x_clean = np.load(fpath).astype(np.float32)

    x_states, deltas, metas = articulation_forward_diffusion_chain_pronounced(
        x_clean=x_clean,
        stats=ARTIC_STATS,
        T=T_STEPS,
        schedule=SCHEDULE,
        seed=SEED + i,
        **FORWARD_CFG
    )
    x_T = x_states[-1]

    x_recovered = reverse_articulation_diffusion(
        model=best_model,
        x_T=x_T,
        T_steps=T_STEPS,
        device=DEVICE
    )

    result = {
        "file": os.path.basename(fpath),
        "clean_to_xT_cos": cosine_similarity_mean(x_clean, x_T),
        "clean_to_rec_cos": cosine_similarity_mean(x_clean, x_recovered),
        "identity_mae": float(np.mean(np.abs(x_clean - x_T))),
        "recovered_mae": float(np.mean(np.abs(x_clean - x_recovered))),
    }
    all_results.append(result)

    print(
        f"[{i:02d}/{len(test_files):02d}] {result['file']} | "
        f"xT cos={result['clean_to_xT_cos']:.6f} | "
        f"rec cos={result['clean_to_rec_cos']:.6f} | "
        f"id mae={result['identity_mae']:.6f} | "
        f"rec mae={result['recovered_mae']:.6f}"
    )

print("\nAverages:")
print("clean -> x_T cosine:", np.mean([r["clean_to_xT_cos"] for r in all_results]))
print("clean -> recovered cosine:", np.mean([r["clean_to_rec_cos"] for r in all_results]))
print("identity MAE:", np.mean([r["identity_mae"] for r in all_results]))
print("recovered MAE:", np.mean([r["recovered_mae"] for r in all_results]))